# Fig 1: Agentic Trace Token Distribution

**Output:** `paper/figures/section3/output/trace_profile.{pdf,png}`

**Step 1 — Start vLLM**  
Serve the model on `localhost:8000` using the config in `config/config.env`:
Model and flags are read from `config/config.env` and `config/model_specific_configs.toml`.

**Step 2 — `input/mini_agent_test.py`**  
Runs `DefaultAgent` over the dataset specified by `DATASET_HF_NAME` (default:
`princeton-nlp/SWE-bench_bm25_13K`). Writes one JSON per problem to
`model_outputs/<MODEL_SHORT>/benchmarks/<BENCHMARK>/swe_output/`.
The swebench config defaults to `step_limit=250`; use `MAX_TURNS` in the run
cell to cap turns. The script runs synchronously and is safe to `%run` from a notebook.

**Step 3 — Compute token stats and plot**  
Reads the per-problem JSONs, computes `uncached_tokens` per turn,
writes `model_outputs/<MODEL_SHORT>/benchmarks/<BENCHMARK>/mini_swe_agent_trace.json`,
and runs `plot_trace.py`.

> **Note on model consistency:** The reference traces were generated with Claude
> (no thinking tokens). Qwen3 models generate `<think>` blocks by default, which
> inflate `completion_tokens`. Keep this in mind when comparing distributions.


In [ ]:
%%bash
cd "$(python3 -c "import pathlib; print(next(p for p in pathlib.Path('.').resolve().parents if (p/'.conserve_root').exists()))")"
source config/config.sh
vllm serve "$MODEL" \
  --download-dir "$MODEL_DIR" --port 8000 \
  "${VLLM_SERVE_FLAGS[@]}" \
  --disable-log-requests


In [ ]:
import json, os, sys
import numpy as np
import pandas as pd
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / '.conserve_root').exists()
)
sys.path.insert(0, str(REPO_ROOT / 'config'))
from config import BENCHMARK, BENCHMARK_TRACE_DIR

# ── Benchmark selection — change these to switch datasets ─────────────────
# BENCHMARK in config/config.env must match DATASET_HF_NAME.split('/')[-1]
DATASET_HF_NAME = 'princeton-nlp/SWE-bench_bm25_13K'
DATASET_SPLIT   = 'test'

HF_CACHE_DIR   = REPO_ROOT / 'conserve' / 'datasets'
SWE_OUTPUT_DIR = BENCHMARK_TRACE_DIR / 'swe_output'
TRACE_PATH     = BENCHMARK_TRACE_DIR / 'mini_swe_agent_trace.json'
PLOT_OUT       = REPO_ROOT / 'paper' / 'figures' / 'section3' / 'output'

print('REPO_ROOT:          ', REPO_ROOT)
print('BENCHMARK:          ', BENCHMARK)
print('BENCHMARK_TRACE_DIR:', BENCHMARK_TRACE_DIR)
print('SWE outputs:        ', SWE_OUTPUT_DIR, '—', len(list(SWE_OUTPUT_DIR.glob('*.json'))), 'files')
print('Trace exists:       ', TRACE_PATH.exists(),
      f'({TRACE_PATH.stat().st_size // 1024 // 1024} MB)' if TRACE_PATH.exists() else '')


REPO_ROOT:           /data/projects/eicchen/conserve_project/conserve
BENCHMARK:           SWE-bench_bm25_13K
BENCHMARK_TRACE_DIR: /data/projects/eicchen/conserve_project/conserve/model_outputs/Qwen3-0.6B/benchmark_trace/SWE-bench_bm25_13K/trace
SWE outputs:         /data/projects/eicchen/conserve_project/conserve/model_outputs/Qwen3-0.6B/benchmark_trace/SWE-bench_bm25_13K/trace/swe_output — 0 files
Trace exists:        False 


## Step 2 — Run mini-SWE-agent

The download cell caches `DATASET_HF_NAME` locally (fast on re-runs).
The run cell executes `mini_agent_test.py` synchronously via `%run` (no async issues).
It writes one JSON per problem to
`model_outputs/<MODEL_SHORT>/benchmarks/<BENCHMARK>/swe_output/`.
Requires vLLM running on `localhost:8000` (Step 1).

**Timing:** ~4 min/problem wall-clock (bash execution dominates LLM call time).
Full 2,294-problem sweep takes ~10–15 h at `MAX_WORKERS=16`.

> **Skip both cells** if `model_outputs/<MODEL_SHORT>/benchmarks/<BENCHMARK>/swe_output/*.json` already exist.


In [8]:
import shutil
from datasets import load_dataset

# Detect stale cache: metadata directory exists but .arrow data files are missing.
# HuggingFace raises FileNotFoundError in this state; clear and re-download.
cache_slug = HF_CACHE_DIR / (DATASET_HF_NAME.replace('/', '___').replace('-', '_').lower())
if cache_slug.exists() and not list(cache_slug.rglob('*.arrow')):
    print(f'Stale cache detected at {cache_slug} (metadata only, no .arrow files). Removing.')
    shutil.rmtree(cache_slug)

print(f'Downloading to {HF_CACHE_DIR} ...')
dataset = load_dataset(
    DATASET_HF_NAME,
    split=DATASET_SPLIT,
    cache_dir=str(HF_CACHE_DIR),
)
print(f'Dataset {DATASET_HF_NAME!r} ready: {len(dataset)} problems')
print('Sample keys:', list(dataset[0].keys())[:6])


Dataset 'princeton-nlp/SWE-bench_bm25_13K' ready: 2294 problems
Sample keys: ['instance_id', 'text', 'repo', 'base_commit', 'problem_statement', 'hints_text']


In [ ]:
# Set limits for quick test runs; leave as None for the full dataset.
MAX_PROBLEMS = 5   # e.g. 10 to run only the first 10 problems
MAX_TURNS    = 5   # e.g. 5 to cap each problem at 5 agent turns

_limit_flags = (
    (f" --max-problems {MAX_PROBLEMS}" if MAX_PROBLEMS is not None else "") +
    (f" --max-turns {MAX_TURNS}"       if MAX_TURNS    is not None else "")
)
%run {str(REPO_ROOT / "conserve" / "input" / "mini_agent_test.py")} --dataset {DATASET_HF_NAME} --split {DATASET_SPLIT}{_limit_flags}

## Step 3 — Compute token stats and plot (Figure 1)

Reads the per-problem JSONs in `model_outputs/<MODEL_SHORT>/benchmarks/<BENCHMARK>/swe_output/`,
computes `uncached_tokens` per turn (new tokens introduced at each step, excluding
the cached prefix), writes a stats-only `mini_swe_agent_trace.json`
(fields: `conv_id`, `iter_id`, `in_token_size`, `out_token_size`),
then runs `plot_trace.py` → Figure 1.


In [ ]:
# Reads per-problem JSONs from mini_agent_test.py output and writes a stats-only
# mini_swe_agent_trace.json. No padding needed — plot_trace.py only uses
# in_token_size and out_token_size.
usages, uncached_tokens_list = [], []
for file_path in sorted(SWE_OUTPUT_DIR.glob('*.json')):
    with open(file_path) as fh:
        data = json.load(fh)
    asst = [m for m in data['messages'] if m['role'] == 'assistant']
    usage = [m['extra']['response']['usage'] for m in asst]
    conv_id = int(file_path.stem.split('_')[-1])
    for i, u in enumerate(usage):
        usage[i]['conv_id'] = conv_id
        usage[i]['iter_id'] = i
    if not usage:
        continue
    usages += usage
    uncached_tokens_list += (
        [usage[0]['prompt_tokens']]
        + [usage[i+1]['prompt_tokens'] - usage[i]['total_tokens']
           for i in range(len(usage) - 1)]
    )
df = pd.DataFrame(usages)
df['uncached_tokens'] = uncached_tokens_list
print(f'Loaded {len(df)} turns from {df["conv_id"].nunique()} conversations')

records = [
    {
        'conv_id':        int(row['conv_id']),
        'iter_id':        int(row['iter_id']),
        'in_token_size':  int(row['uncached_tokens']),
        'out_token_size': int(row['completion_tokens']),
    }
    for _, row in df.iterrows()
]
records.sort(key=lambda r: (r['conv_id'], r['iter_id']))
TRACE_PATH.parent.mkdir(parents=True, exist_ok=True)
TRACE_PATH.write_text(json.dumps(records))
print(f'Saved {len(records)} stats-only records → {TRACE_PATH}')


In [ ]:
%matplotlib inline
%run ../scripts/plot_trace.py


In [ ]:
from IPython.display import Image
Image(str(PLOT_OUT / 'trace_profile.png'))
